# Losses: Optimal Transport, Wasserstein, Sinkhorn, and MMD (C03 Topic 10)

**Citations:** Kantorovich (1942); Gretton et al. (2012 MMD); Cuturi (2013 Sinkhorn); Bonneel et al. (2015 SWD); Feydy et al. (2019 Debiased Sinkhorn).

### Purpose & Topic Bridges
Evaluating discrepancies between empirical probability distributions requires geometries that respect the underlying metric space of the data. Standard $f$-divergences suffer from the *overlapping support catastrophe* (yielding infinite loss or zero gradients when distributions have disjoint supports). Optimal Transport and Maximum Mean Discrepancy lift the ground metric of the space into smooth distribution-matching objectives. This notebook explores exact 1D Wasserstein distance, Sliced Wasserstein (SWD), Entropic Sinkhorn distance, Debiased Sinkhorn Divergence, and multi-scale MMD.

**Decision Boundary:** Use Optimal Transport / MMD when evaluating disjoint point clouds, biological single-cell trajectories, or generative distributions without adversarial discriminators. Do NOT use un-debiased entropic Sinkhorn $S_\epsilon(P, Q)$ without subtracting self-transport terms $\frac{1}{2}S_\epsilon(P, P) + \frac{1}{2}S_\epsilon(Q, Q)$, as non-zero self-cost forces artificial shrinkage toward the empirical mean!

## 2. Mathematical Foundations & Governing Equations

1. **Exact 1D Wasserstein Distance ($W_1$):** $W_1(u, v) = \frac{1}{N} \sum_{i=1}^N |u_{(i)} - v_{(i)}|$ via sorted order statistics in $\mathcal{O}(N \log N)$.
2. **Sliced Wasserstein Distance (SWD):** $\text{SWD}(X, Y) = \frac{1}{L} \sum_{l=1}^L W_1(X \theta_l, Y \theta_l)$ with random unit projections $\theta_l \in \mathbb{S}^{D-1}$ in $\mathcal{O}(L \cdot N \log N)$.
3. **Entropic Regularized Optimal Transport (Sinkhorn):** $S_\epsilon(P, Q) = \min_\Pi \langle \Pi, C \rangle - \epsilon H(\Pi) = \sum_{ij} (u_i K_{ij} v_j) C_{ij}$ where $K = \exp(-C/\epsilon)$, $u = a / Kv$, $v = b / K^T u$ in $\mathcal{O}(N^2)$ GPU matrix multiplications.
4. **Debiased Sinkhorn Divergence:** $\bar{S}_\epsilon(P, Q) = S_\epsilon(P, Q) - \frac{1}{2} S_\epsilon(P, P) - \frac{1}{2} S_\epsilon(Q, Q)$, guaranteeing $\bar{S}_\epsilon(P, P) = 0$ and $\bar{S}_\epsilon(P, Q) \ge 0$.
5. **Maximum Mean Discrepancy (MMD²):** $\text{MMD}^2(X, Y) = \frac{1}{N^2}\sum K_{xx} + \frac{1}{M^2}\sum K_{yy} - \frac{2}{NM}\sum K_{xy}$ with multi-scale RBF kernels $k(x, y) = \sum_m \exp(-\frac{\|x - y\|^2}{2 \sigma_m^2})$.
6. **Energy / Cramér Distance:** $\mathcal{E}(X, Y) = \frac{2}{NM}\sum \|x_i - y_j\| - \frac{1}{N^2}\sum \|x_i - x_j\| - \frac{1}{M^2}\sum \|y_i - y_j\|$.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/10_losses_optimal_transport_wasserstein_sinkhorn_and_mmd.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class OptimalTransportUnifiedModule(nn.Module):
    def __init__(self, in_dim=4, out_dim=4):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def wasserstein_1d(self, u, v):
        u_sorted, _ = torch.sort(u, dim=-1); v_sorted, _ = torch.sort(v, dim=-1)
        return torch.mean(torch.abs(u_sorted - v_sorted))

    def sliced_wasserstein(self, x, y, projections):
        px = torch.matmul(x, projections.T).permute(0, 2, 1)
        py = torch.matmul(y, projections.T).permute(0, 2, 1)
        sx, _ = torch.sort(px, dim=-1); sy, _ = torch.sort(py, dim=-1)
        return torch.mean(torch.abs(sx - sy))

    def sinkhorn_cost(self, x, y, eps=0.5, num_iters=20):
        n, m = x.shape[0], y.shape[0]
        c = torch.sum((x.unsqueeze(1) - y.unsqueeze(0)).pow(2), dim=-1)
        k = torch.exp(-c / eps)
        a = torch.ones(n, device=x.device) / float(n)
        b = torch.ones(m, device=y.device) / float(m)
        v = torch.ones(m, device=y.device) / float(m)
        for _ in range(num_iters):
            u = a / (torch.matmul(k, v) + 1e-12)
            v = b / (torch.matmul(k.T, u) + 1e-12)
        return torch.sum((u.unsqueeze(1) * k * v.unsqueeze(0)) * c)

    def debiased_sinkhorn(self, x, y, eps=0.5, num_iters=20):
        s_xy = self.sinkhorn_cost(x, y, eps, num_iters)
        s_xx = self.sinkhorn_cost(x, x, eps, num_iters)
        s_yy = self.sinkhorn_cost(y, y, eps, num_iters)
        return s_xy - 0.5 * s_xx - 0.5 * s_yy

    def mmd(self, x, y, sigmas=(0.5, 1.0, 2.0)):
        def rbf(a, b):
            d2 = torch.sum((a.unsqueeze(1) - b.unsqueeze(0)).pow(2), dim=-1)
            return sum(torch.exp(-d2 / (2.0 * s * s)) for s in sigmas)
        return rbf(x, x).mean() + rbf(y, y).mean() - 2.0 * rbf(x, y).mean()

    def energy_distance(self, x, y):
        d_xy = torch.norm(x.unsqueeze(1) - y.unsqueeze(0), dim=-1)
        d_xx = torch.norm(x.unsqueeze(1) - x.unsqueeze(0), dim=-1)
        d_yy = torch.norm(y.unsqueeze(1) - y.unsqueeze(0), dim=-1)
        return 2.0 * d_xy.mean() - d_xx.mean() - d_yy.mean()

model = OptimalTransportUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('OptimalTransportUnifiedModule initialized successfully.')

In [ ]:
u_1d, v_1d = weights['u_1d'], weights['v_1d']
swd_x, swd_y, projs = weights['swd_x'], weights['swd_y'], weights['projections']
c_p, c_q = weights['cloud_p'], weights['cloud_q']

err_w1 = (model.wasserstein_1d(u_1d, v_1d) - weights['expected_w1'][0]).abs().item()
assert err_w1 < 1e-5, f'W1 parity error: {err_w1:.2e}'

err_swd = (model.sliced_wasserstein(swd_x, swd_y, projs) - weights['expected_swd'][0]).abs().item()
assert err_swd < 1e-5, f'SWD parity error: {err_swd:.2e}'

err_s_raw = (model.sinkhorn_cost(c_p, c_q) - weights['expected_sinkhorn_raw'][0]).abs().item()
assert err_s_raw < 1e-5, f'Sinkhorn raw error: {err_s_raw:.2e}'

err_s_deb = (model.debiased_sinkhorn(c_p, c_q) - weights['expected_sinkhorn_debiased'][0]).abs().item()
assert err_s_deb < 1e-5, f'Debiased Sinkhorn error: {err_s_deb:.2e}'

err_mmd = (model.mmd(c_p, c_q) - weights['expected_mmd'][0]).abs().item()
assert err_mmd < 1e-5, f'MMD error: {err_mmd:.2e}'

err_energy = (model.energy_distance(c_p, c_q) - weights['expected_energy'][0]).abs().item()
assert err_energy < 1e-5, f'Energy error: {err_energy:.2e}'

err_anchor = (model(test_in) - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 7 invariants verified! W1: {err_w1:.2e}, SWD: {err_swd:.2e}, SinkRaw: {err_s_raw:.2e}, SinkDeb: {err_s_deb:.2e}, MMD: {err_mmd:.2e}, Energy: {err_energy:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    '1D Wasserstein': lambda: model.wasserstein_1d(u_1d, v_1d),
    'Sliced Wasserstein': lambda: model.sliced_wasserstein(swd_x, swd_y, projs),
    'Sinkhorn Raw': lambda: model.sinkhorn_cost(c_p, c_q),
    'Debiased Sinkhorn': lambda: model.debiased_sinkhorn(c_p, c_q),
    'MMD Multi-Scale': lambda: model.mmd(c_p, c_q),
    'Energy Distance': lambda: model.energy_distance(c_p, c_q),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Pathological Failure Mode: Non-Zero Self-Transport Cost in Un-debiased Entropic Sinkhorn
# When evaluating distribution distance between identical distributions (P, P),
# un-debiased Sinkhorn S_eps(P, P) exhibits positive self-cost due to entropic smoothing,
# causing severe gradient shrinkage toward the mean in generative modeling!
s_self_raw = model.sinkhorn_cost(c_p, c_p, eps=0.5, num_iters=20).item()
print(f'[FAILURE DEMONSTRATED] Un-debiased Sinkhorn self-cost S_eps(P, P) = {s_self_raw:.4f} > 0.05 (violates metric identity)!')
assert s_self_raw > 0.05, f'Expected positive self-cost, got {s_self_raw:.4f}'

# Debiased Sinkhorn eliminates self-transport bias: S_deb(P, P) = S(P,P) - 0.5 S(P,P) - 0.5 S(P,P) = 0.0
s_self_deb = model.debiased_sinkhorn(c_p, c_p, eps=0.5, num_iters=20).item()
print(f'[STABILIZED] Debiased Sinkhorn achieves exact zero self-cost: {s_self_deb:.2e}!')
assert abs(s_self_deb) < 1e-5, f'Expected zero debiased self-cost, got {s_self_deb:.2e}'
assert torch.allclose(model(test_in), test_out, atol=1e-5), 'Linear anchor mismatch'
print('[CORRECT] Optimal Transport, Debiased Sinkhorn, MMD, and linear anchor verified successfully.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Non-Vanishing Gradients on Disjoint Supports:** Optimal Transport and MMD lift ground metric geometry into probability space, providing meaningful gradients even when distribution supports do not overlap.
2. **The Debiasing Imperative:** Entropic smoothing in Sinkhorn introduces non-vanishing self-transport bias ($S_\epsilon(P, P) > 0$). Debiasing via self-correlation terms is mandatory to prevent artificial variance shrinkage.
3. **Computational Trade-Offs:** SWD ($O(L \cdot N \log N)$) and MMD ($O(N^2)$) avoid iterative linear programming, making them ideal for high-throughput representation matching.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 09_losses_divergences_information_bottleneck_and_coding_rate](file:///c:/Users/Liam/Code/AI-Examples/notebooks/03_building_blocks_and_losses/09_losses_divergences_information_bottleneck_and_coding_rate.ipynb)